In [3]:
# load env variables & create client
from dotenv import load_dotenv

load_dotenv()

# create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

def add_user_message(messages, text):
    user_message = { "role": "user", "content": text}
    messages.append(user_message)
def add_assistant_message(messages, text):
    assistant_message = { "role": "assistant", "content": text}
    messages.append(assistant_message)


#when using streaming, the chat function works a bit differently, so we're now manually calling 
# the client.messages.create function in the next part of the notebook
def chat(messages, system=None, temperature=1.0):
    params = {
        "model":model,
        "max_tokens":1000,
        "messages": messages,
        "temperature": temperature
    }

#This approach handles an important detail: Claude's API doesn't accept system=None, 
# so you need to conditionally include the system parameter only when it's provided.
    if system:
        params["system"] = system
    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    #boo! This prints all blocks, we only need the content blocks
    print(event)

RawMessageStartEvent(message=Message(id='msg_01WAUUjyniHWF78RbCcafZ8r', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='"', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='CustomerVault is a fictional relational database containing fabricated records', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' of 10,000 imaginary custo

In [ ]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
#yay anthropic sdk
) as stream:
    for text in stream.text_stream:
       # print(text, end="")
        pass
    # Get the complete message for storage
    final_message = stream.get_final_message()

TypeError: 'tuple' object is not callable